In [1]:
import math
import multiprocessing
import os

from multiprocessing import Pool


import joblib
from joblib import Parallel, delayed, parallel_backend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


import fnmatch
import sys
import itertools


#from sklearn.metrics import mean_squared_error, r2_score
#from sklearn.preprocessing import StandardScaler


from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

import cartopy.feature as cfeature
import dask


#import joblib

import xarray as xr
#from tqdm.notebook import tqdm



In [2]:
%run ./common_vars.py

%run ~/repos/spatialtuning/funs/plotting.py
%run ~/repos/spatialtuning/funs/utils.py


In [3]:
para = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/paras_v4.csv", index_col=0)
ppe_pd = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/ppe_zonal_v4_iteration1.csv", index_col=0)
obs_pd = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/obs_zonal_v4_iteration1.csv", index_col=0)

In [4]:
cli_nm = list(obs_dict.keys())
para_nm = list(para.columns)


In [5]:
para_norm = para.copy()
para_norm = (para_norm - para_norm.min())/(para_norm.max() - para_norm.min())


In [6]:
para_emu = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/para_samples.nc").to_dataframe()


In [11]:
from joblib import Parallel, delayed

results = Parallel(n_jobs=16)(
    delayed(gp_training_application)(para_norm, ppe_pd, y_name, para_emu, path = "/glade/work/qingyuany/repo_data/spatialtuning/", n_sens_p=2)
    for y_name in list(ppe_pd.columns)
)




/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 0.8. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 0.8. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/glade/work/qingyuany/conda-envs/ml_env/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 0.8. Increasing the bound and calling fit again may find a better value.
  warnings.warn

In [12]:
results

[['SWCF_zonal_lat_-82', array([32, 30])],
 ['SWCF_zonal_lat_-77', array([32, 31])],
 ['SWCF_zonal_lat_-72', array([32, 29])],
 ['SWCF_zonal_lat_-67', array([24,  8])],
 ['SWCF_zonal_lat_-62', array([24,  8])],
 ['SWCF_zonal_lat_-57', array([24,  8])],
 ['SWCF_zonal_lat_-52', array([24,  8])],
 ['SWCF_zonal_lat_-47', array([ 8, 29])],
 ['SWCF_zonal_lat_-42', array([ 8, 29])],
 ['SWCF_zonal_lat_-37', array([ 8, 29])],
 ['SWCF_zonal_lat_-32', array([ 8, 29])],
 ['SWCF_zonal_lat_-27', array([ 8, 29])],
 ['SWCF_zonal_lat_-22', array([ 8, 29])],
 ['SWCF_zonal_lat_-17', array([ 8, 32])],
 ['SWCF_zonal_lat_-12', array([ 8, 32])],
 ['SWCF_zonal_lat_-7', array([ 8, 20])],
 ['SWCF_zonal_lat_-2', array([20,  8])],
 ['SWCF_zonal_lat_2', array([20,  8])],
 ['SWCF_zonal_lat_7', array([20,  8])],
 ['SWCF_zonal_lat_12', array([32,  8])],
 ['SWCF_zonal_lat_17', array([ 8, 32])],
 ['SWCF_zonal_lat_22', array([ 8, 32])],
 ['SWCF_zonal_lat_27', array([ 8, 29])],
 ['SWCF_zonal_lat_32', array([ 8, 29])],
 ['

In [9]:
# from joblib import parallel_backend
# with parallel_backend('loky', inner_max_num_threads=1):
#     results = Parallel(n_jobs=16)(
#         delayed(gp_training_application)(para_norm, ppe_pd, y_name, para_emu, no_sen=3)
#         for y_name in list(ppe_pd.columns)
#     )

In [14]:
cli_sensitive_para = {pair[0]: pd.Series(pair[1]) for pair in results if pair is not None}
cli_sensitive_para_pd = pd.concat(list(cli_sensitive_para.values()), axis = 1)
cli_sensitive_para_pd.columns = list(cli_sensitive_para.keys())
#cli_sensitive_para_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/meta_cli_para.csv")